In [1]:
import pandas as pd

In [2]:
reader = pd.io.stata.StataReader("data/issp2020/raw/ZA7650_v2-0-0.dta")
df = reader.read(convert_categoricals=False)
var_labels, val_labels = reader.variable_labels(), reader.value_labels()
var_labels


{'studyno': 'GESIS Study Number',
 'version': 'GESIS Archive version',
 'doi': 'Digital Object Identifier',
 'country': 'Country ISO 3166 Code (see c_sample for codes for the sample)',
 'c_sample': 'Country/ Sample ISO 3166 Code (see country for codes for whole nation states)',
 'c_alphan': 'Country/ Sample Prefix ISO 3166 Code - alphanumeric',
 'v1': "Q1a Most important issue for [R's COUNTRY] today?",
 'v2': "Q1b Next most important issue for [R's COUNTRY] today?",
 'v3': 'Q2a Solve economic problems: private enterprise',
 'v4': 'Q2b Responsibility of government: reduce income differences',
 'v5': 'Q2c [COUNTRY] should limit import of foreign products to protect national econom',
 'v6': 'Q2d [COUNTRY] should limit immigration to protect national way of life',
 'v7': 'Q2e International organizations are taking away too much power from the governme',
 'v8': "Q3a Highest priority in [R's COUNTRY]?",
 'v9': "Q3b Next highest priority in [R's COUNTRY]?",
 'v10': 'Q4 Amount of trust in mos

In [16]:
# val_labels
df.iloc[0].v1

np.int8(1)

In [12]:

targets = [f"v{i}" for i in range(3, 61) if i not in (8, 9, 48, 49, 51)] + ["v1", "v8"]
persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "MAINSTAT", "ISCO08", "MARITAL", "PARTLIV",
           "HOMPOP", "HHCHILDR", "HHTODD", "URBRURAL", "RELIGGRP", "ATTEND", "TOPBOT", "PARTY_LR"]
reg_cols = [c for c in df.columns if c.endswith("_REG")]
inc_cols = [c for c in df.columns if c.endswith("_INC")]

num = targets + persona[1:] + reg_cols + inc_cols
df[num] = df[num].where(df[num] >= 0)  # negative codes -> NaN

# Dataset preparation

Decisions are in `data/issp2020/DECISIONS.md`. These cells load their own copy of the data (`raw`).

In [7]:
import subprocess
import pandas as pd

rd = pd.io.stata.StataReader("data/issp2020/raw/ZA7650_v2-0-0.dta")
raw = rd.read(convert_categoricals=False)
VAL_LABELS, LBL = rd.value_labels(), dict(zip(rd._varlist, rd._lbllist))
labels = lambda col: VAL_LABELS.get(LBL[col], {})
valid = lambda s: s.notna() & (s >= 0)  # negative codes = missing
raw.shape

(44100, 337)

## Question bank

In [8]:
subprocess.run(["python3", "data/issp2020/build_question_bank.py"], check=True)
qb = pd.read_csv("data/issp2020/question_bank.csv")
qb[["v", "type", "question", "options"]]

wrote 60 items to ISSP2020/question_bank.csv


,v,type,question,options
0,v1,choice,Which of these issues is the most important fo...,Health care | Education | Crime | The environm...
1,v2,exclude,Which is the next most important?,Health care | Education | Crime | The environm...
2,v3,score,How much do you agree or disagree with each of...,Disagree strongly | Disagree | Neither agree n...
3,v4,score,How much do you agree or disagree with each of...,Disagree strongly | Disagree | Neither agree n...
4,v5,score,How much do you agree or disagree with each of...,Disagree strongly | Disagree | Neither agree n...
5,v6,score,How much do you agree or disagree with each of...,Disagree strongly | Disagree | Neither agree n...
6,v7,score,How much do you agree or disagree with each of...,Disagree strongly | Disagree | Neither agree n...
7,v8,choice,"Looking at the list below, please tick a box n...",Maintain order in the nation | Give people mor...
8,v9,exclude,And which one do you think should be {country}...,Maintain order in the nation | Give people mor...
9,v10,score,"Generally speaking, would you say that most pe...",1 You can't be too careful | 2 | 3 | 4 | 5 Mos...


## Persona fields: coverage by country (% valid)

In [9]:
bg = ["SEX", "AGE", "EDULEVEL", "WORK", "MAINSTAT", "MARITAL", "HOMPOP", "RELIGGRP", "ATTEND", "URBRURAL",
      "TOPBOT", "PARTLIV", "HHCHILDR", "UNION", "PARTY_LR"]
coverage = pd.DataFrame({c: raw.groupby("c_alphan")[c].apply(lambda s: valid(s).mean()) for c in bg}).T
(coverage * 100).round().astype(int)

c_alphan,AT,AU,CH,CN,DE,DK,ES,FI,FR,HR,...,NZ,PH,RU,SE,SI,SK,TH,TW,US,ZA
SEX,100,98,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,100,100,100,100
AGE,100,96,99,100,99,88,99,100,100,99,...,98,100,100,100,100,100,100,100,94,100
EDULEVEL,100,97,97,100,99,92,98,99,93,99,...,98,100,100,95,99,100,100,100,99,99
WORK,100,99,98,100,98,97,99,100,100,97,...,98,100,99,93,99,100,99,100,100,93
MAINSTAT,100,97,97,96,97,97,95,100,100,96,...,98,100,100,94,99,98,98,100,100,97
MARITAL,100,96,99,98,98,86,97,99,92,99,...,97,100,100,94,99,97,88,99,100,97
HOMPOP,100,94,92,100,98,88,94,99,100,96,...,100,100,100,95,99,100,98,100,100,100
RELIGGRP,99,98,98,100,97,90,96,98,99,98,...,97,100,100,95,99,93,100,100,99,97
ATTEND,99,98,98,100,99,85,96,99,92,96,...,96,100,99,96,96,91,94,100,99,96
URBRURAL,100,97,98,0,100,89,98,99,93,100,...,98,100,100,95,100,100,100,100,100,100


## Persona group sizes

Respondents with identical personas form one group; its answer shares are the target. More fields → smaller groups → noisier targets.

In [ ]:
p = pd.DataFrame({"country": raw.c_alphan})
p["sex"] = raw.SEX.where(valid(raw.SEX))
p["age"] = pd.cut(raw.AGE.where(valid(raw.AGE)), [0, 29, 44, 59, 74, 120], labels=["15-29", "30-44", "45-59", "60-74", "75+"])
p["edu"] = pd.cut(raw.EDULEVEL.where(valid(raw.EDULEVEL)), [-1, 2, 4, 8], labels=["low", "medium", "high"])
p["status"] = raw.MAINSTAT.where(valid(raw.MAINSTAT)).map({1: "working", 2: "unemployed", 3: "student", 4: "student",
                                                          6: "retired", 7: "homemaker"}).fillna("other")
p["urban"] = raw.URBRURAL.where(valid(raw.URBRURAL)).map({1: "city", 2: "city", 3: "town", 4: "rural", 5: "rural"})
p["religious"] = pd.cut(raw.ATTEND.where(valid(raw.ATTEND)), [0, 4, 6, 8], labels=["monthly+", "rarely", "never"])

sets = {
    "A: country, sex, age": ["country", "sex", "age"],
    "B: A + education": ["country", "sex", "age", "edu"],
    "C: B + main status": ["country", "sex", "age", "edu", "status"],
    "D: B + urban/rural": ["country", "sex", "age", "edu", "urban"],
    "E: C + urban/rural": ["country", "sex", "age", "edu", "status", "urban"],
    "F: E + religious attendance": ["country", "sex", "age", "edu", "status", "urban", "religious"],
}
rows = []
for name, cols in sets.items():
    n = p.dropna(subset=cols).groupby(cols, observed=True).size()
    rows.append({"persona": name, "respondents": n.sum(), "groups": len(n), "median size": n.median(),
                 **{f"% in groups ≥{k}": round(100 * n[n >= k].sum() / n.sum()) for k in (10, 20, 30)}})
pd.DataFrame(rows).set_index("persona")

## Build the dataset (persona B)

In [11]:
subprocess.run(["python3", "data/issp2020/build_dataset.py"], check=True)
ds = pd.read_json("data/issp2020/issp2020.jsonl", lines=True)
ds.head()

13981 records, 36518 question targets, from 42932 respondents in 820 persona groups -> ISSP2020/issp2020_kev.jsonl


,state,questions,meta
0,Country: Austria\nSex: Female\nAge: 15-29\nEdu...,"{'v1': {'type': 'choice', 'criteria': {'Health...","{'country': 'AT', 'sex': 'Female', 'age': '15-..."
1,Country: Austria\nSex: Female\nAge: 15-29\nEdu...,"{'v4': {'type': 'score', 'criteria': ['Disagre...","{'country': 'AT', 'sex': 'Female', 'age': '15-..."
2,Country: Austria\nSex: Female\nAge: 15-29\nEdu...,"{'v8': {'type': 'choice', 'criteria': {'Mainta...","{'country': 'AT', 'sex': 'Female', 'age': '15-..."
3,Country: Austria\nSex: Female\nAge: 15-29\nEdu...,"{'v10': {'type': 'score', 'criteria': ['1 You ...","{'country': 'AT', 'sex': 'Female', 'age': '15-..."
4,Country: Austria\nSex: Female\nAge: 15-29\nEdu...,"{'v12': {'type': 'score', 'criteria': ['0 No t...","{'country': 'AT', 'sex': 'Female', 'age': '15-..."


## Load the Kev dataset

`records`: the raw JSONL records. `kev`: one row per persona group × question.

In [14]:
import json

records = [json.loads(line) for line in open("data/issp2020/issp2020.jsonl", encoding="utf-8")]

# one row per (persona group, question)
rows = []
for r in records:
    for v, q in r["questions"].items():
        rows.append({"country": r["country_code"], **{k.lower(): x for k, x in r["persona"].items() if k != "Country"},
                     "block": r["block"], "v": v, "type": q["type"], "n": r["n"][v], "label": q["label"],
                     "target": q["target"], "criteria": q["criteria"], "instructions": q["instructions"]})
kev = pd.DataFrame(rows)
print(f"{len(records)} records, {len(kev)} question targets")
kev

13981 records, 36518 question targets


,country,sex,age,education,block,v,type,n,label,target,criteria,instructions,state
0,AT,Female,15-29,Lower secondary or less,Q1,v1,choice,10,Health care,"{'Health care': 0.4, 'Education': 0.2, 'The en...","{'Health care': None, 'Education': None, 'Crim...",Which of these issues is the most important fo...,Country: Austria\nSex: Female\nAge: 15-29\nEdu...
1,AT,Female,15-29,Lower secondary or less,Q2,v4,score,11,3,"{'3': 0.5455, '4': 0.4545}","[Disagree strongly, Disagree, Neither agree no...",How much do you agree or disagree with each of...,Country: Austria\nSex: Female\nAge: 15-29\nEdu...
2,AT,Female,15-29,Lower secondary or less,Q2,v6,score,10,1,"{'1': 0.4, '0': 0.3, '4': 0.2, '2': 0.1}","[Disagree strongly, Disagree, Neither agree no...",How much do you agree or disagree with each of...,Country: Austria\nSex: Female\nAge: 15-29\nEdu...
3,AT,Female,15-29,Lower secondary or less,Q3,v8,choice,11,Protect freedom of speech,"{'Protect freedom of speech': 0.3636, 'Fight r...","{'Maintain order in the nation': None, 'Give p...","Looking at the list below, please tick a box n...",Country: Austria\nSex: Female\nAge: 15-29\nEdu...
4,AT,Female,15-29,Lower secondary or less,Q4,v10,score,11,2,"{'2': 0.3636, '4': 0.2727, '1': 0.2727, '3': 0...","[1 You can't be too careful, 2, 3, 4, 5 Most p...","Generally speaking, would you say that most pe...",Country: Austria\nSex: Female\nAge: 15-29\nEdu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36513,ZA,Male,75 or older,Upper secondary or post-secondary non-tertiary,Q21,v56,noul,11,False,{'false': 1.0},"{'true': 'Yes, I have', 'false': 'No, I have n...","In the last five years, have you given money t...",Country: South Africa\nSex: Male\nAge: 75 or o...
36514,ZA,Male,75 or older,Upper secondary or post-secondary non-tertiary,Q21,v57,noul,11,False,{'false': 1.0},"{'true': 'Yes, I have', 'false': 'No, I have n...","In the last five years, have you taken part in...",Country: South Africa\nSex: Male\nAge: 75 or o...
36515,ZA,Male,75 or older,Upper secondary or post-secondary non-tertiary,Q22,v58,score,11,0,"{'0': 0.4545, '2': 0.2727, '1': 0.1818, '3': 0...","[Not at all, To a small extent, To some extent...","Thinking about your neighbourhood, to what ext...",Country: South Africa\nSex: Male\nAge: 75 or o...
36516,ZA,Male,75 or older,Upper secondary or post-secondary non-tertiary,Q22,v59,score,11,1,"{'1': 0.3636, '0': 0.3636, '2': 0.1818, '3': 0...","[Not at all, To a small extent, To some extent...","Thinking about your neighbourhood, to what ext...",Country: South Africa\nSex: Male\nAge: 75 or o...


In [16]:
kev.iloc[0]

country                                                        AT
sex                                                        Female
age                                                         15-29
education                                 Lower secondary or less
block                                                          Q1
v                                                              v1
type                                                       choice
n                                                              10
label                                                 Health care
target          {'Health care': 0.4, 'Education': 0.2, 'The en...
criteria        {'Health care': None, 'Education': None, 'Crim...
instructions    Which of these issues is the most important fo...
state           Country: Austria\nSex: Female\nAge: 15-29\nEdu...
Name: 0, dtype: object

In [ ]:
kev.iloc[0]['instructions']


'Which of these issues is the most important for Austria today?'

In [21]:
print(kev.iloc[0]['state'])


Country: Austria
Sex: Female
Age: 15-29
Education: Lower secondary or less


In [22]:
print(kev.iloc[0]['instructions'])


Which of these issues is the most important for Austria today?


In [17]:
kev.iloc[0]['label']

'Health care'

In [23]:
kev.iloc[0]['target']


{'Health care': 0.4,
 'Education': 0.2,
 'The environment': 0.2,
 'Crime': 0.1,
 'None of these': 0.1}

In [24]:
kev.iloc[0]['criteria']


{'Health care': None,
 'Education': None,
 'Crime': None,
 'The environment': None,
 'Immigration': None,
 'The economy': None,
 'Terrorism': None,
 'Poverty': None,
 'None of these': None}

## Sample model inputs and outputs, one per question type

Input rendered by Kev's own `to_record()`; output = the target the model is trained to match.

In [27]:
import sys
sys.path.insert(0, "kev")
from kev.api import SystemOneRequest, to_record

def model_input(state, v, q):
    """The text Kev's model reads for one question, rendered by Kev's own to_record().
    Each question is its own branch: it sees the state and itself, not the other questions.
    Kev's delimiter tokens are shown as <state>, <q>, <opt>...</opt>, <decide>."""
    req = SystemOneRequest.model_validate({"state": state, "questions": {
        v: {"type": q["type"], "instructions": q["instructions"], "criteria": q["criteria"]}}})
    rec, meta = to_record(req)
    (qr,), (m,) = rec["questions"], meta
    text = f"<state>{rec['state']}\n<q>{qr['instr']}\n" + "".join(f"<opt>{o}</opt>\n" for o in qr["options"]) + "<decide>"
    return text, m["keys"], qr["options"]

def show_example(qtype):
    r = next(r for r in records if any(q["type"] == qtype for q in r["questions"].values()))
    v, q = next((v, q) for v, q in r["questions"].items() if q["type"] == qtype)
    state = "\n".join(f"{k}: {x}" for k, x in r["persona"].items())
    text, keys, options = model_input(state, v, q)
    print(f"===== {qtype}: {v}  (n = {r['n'][v]} respondents) =====")
    print("INPUT:\n" + text)
    print("\nOUTPUT (target probabilities, in option order):")
    for k, o in zip(keys, options):
        print(f"  {q['target'].get(k, 0.0):6.3f}  {o}")
    print(f"label (most common answer): {q['label']!r}\n")

for t in ["score", "choice", "noul"]:
    show_example(t)

===== score: v4  (n = 11 respondents) =====
INPUT:
<state>Country: Austria
Sex: Female
Age: 15-29
Education: Lower secondary or less
<q>How much do you agree or disagree with each of these statements? It is the responsibility of the government to reduce the differences in income between people with high incomes and those with low incomes.
<opt>Disagree strongly</opt>
<opt>Disagree</opt>
<opt>Neither agree nor disagree</opt>
<opt>Agree</opt>
<opt>Agree strongly</opt>
<decide>

OUTPUT (target probabilities, in option order):
   0.000  Disagree strongly
   0.000  Disagree
   0.000  Neither agree nor disagree
   0.545  Agree
   0.455  Agree strongly
label (most common answer): 3

===== choice: v1  (n = 10 respondents) =====
INPUT:
<state>Country: Austria
Sex: Female
Age: 15-29
Education: Lower secondary or less
<q>Which of these issues is the most important for Austria today?
<opt>Health care</opt>
<opt>Education</opt>
<opt>Crime</opt>
<opt>The environment</opt>
<opt>Immigration</opt>
<opt